In [1]:
from astroquery.gaia import Gaia
import pandas as pd
from datapaths import Datapaths

The archive is unstable and may perform below expectations. If launching multiple, consecutive, heavy queries through Python, please space them out (e.g., using sleep(1)) to avoid overloading the system. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the December 2025 infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
dp = Datapaths()

In [3]:
dp.print_paths()

name          | path                                                             | tags       | type | updated at               
--------------+------------------------------------------------------------------+------------+------+--------------------------
dr3_sample_v1 | /home/alex/Data/Work/Sources/gaia_vis/data/dr3_sample_v1.parquet | dr3,sample | data | 2026-03-12T21:07:01+01:00


### Inspect Gaia schema

In [38]:
# Query params
TABLE = "gaiadr3.gaia_source"
COLUMNS = [
    "source_id",
    "ra", "ra_error", "dec", "dec_error", "parallax", "parallax_error", "parallax_over_error", 
    "pm", "pmra", "pmra_error", "pmdec", "pmdec_error", "phot_g_mean_flux", "phot_g_mean_flux_error", 
    "phot_g_mean_flux_over_error", "phot_g_mean_mag", "phot_bp_mean_flux", "phot_bp_mean_flux_error", 
    "phot_bp_mean_flux_over_error", "phot_bp_mean_mag", "phot_rp_mean_flux", "phot_rp_mean_flux_error", 
    "phot_rp_mean_flux_over_error", "phot_rp_mean_mag", "bp_rp", "bp_g", "g_rp", "radial_velocity", 
    "radial_velocity_error", "phot_variable_flag", "l", "b", "ecl_lon", "ecl_lat", "in_qso_candidates", 
    "in_galaxy_candidates", "non_single_star", "teff_gspphot", "teff_gspphot_lower", "teff_gspphot_upper", 
    "logg_gspphot", "logg_gspphot_lower", "logg_gspphot_upper", "mh_gspphot", "mh_gspphot_lower", 
    "mh_gspphot_upper", "distance_gspphot", "distance_gspphot_lower", "distance_gspphot_upper"
]
N_SAMPLE = 120000 # slightly larger sample size because Bailer-Jones table has fewer rows than the main Gaia table

In [30]:
# Inspect schema for Gaia DR3
tab = Gaia.load_table(TABLE)
colnames = [c.name for c in tab.columns]
coldescr = [c.description for c in tab.columns]
coltype = [c.data_type for c in tab.columns]
colunit = [c.unit for c in tab.columns]
columns_info = pd.DataFrame.from_dict({
    'name': colnames, 'description': coldescr, 'type': coltype, 'unit': colunit, 'in_sample': [c in COLUMNS for c in colnames]
})


In [23]:
columns_info[columns_info['in_sample']].style.set_properties(
    **{
        'inline-size': '500px',
        'overflow-wrap': 'break-word',
    }, 
    subset='description'
)

,name,description,type,unit,in_sample
2,source_id,Unique source identifier (unique within a particular Data Release),long,nan,True
5,ra,Right ascension,double,deg,True
6,ra_error,Standard error of right ascension,float,mas,True
7,dec,Declination,double,deg,True
8,dec_error,Standard error of declination,float,mas,True
9,parallax,Parallax,double,mas,True
10,parallax_error,Standard error of parallax,float,mas,True
11,parallax_over_error,Parallax divided by its standard error,float,nan,True
12,pm,Total proper motion,float,mas.yr**-1,True
13,pmra,Proper motion in right ascension direction,double,mas.yr**-1,True


In [31]:
# Query params
TABLE_BJ = "external.gaiaedr3_distance"
COLUMNS_BJ = [
    "source_id",
    "r_med_geo", "r_lo_geo", "r_hi_geo", "r_med_photogeo", "r_lo_photogeo", "r_hi_photogeo", "flag"
    ]

In [33]:
# Inspect schema for Gaia Bailer-Jones distances
tab_bj = Gaia.load_table(TABLE_BJ)
colnames_bj = [c.name for c in tab_bj.columns]
coldescr_bj = [c.description for c in tab_bj.columns]
coltype_bj = [c.data_type for c in tab_bj.columns]
colunit_bj = [c.unit for c in tab_bj.columns]
columns_info_bj = pd.DataFrame.from_dict({
    'name': colnames_bj, 'description': coldescr_bj, 'type': coltype_bj, 'unit': colunit_bj, 'in_sample': [c in COLUMNS_BJ for c in colnames_bj]
})


In [34]:
columns_info_bj[columns_info_bj['in_sample']].style.set_properties(
    **{
        'inline-size': '500px',
        'overflow-wrap': 'break-word',
    }, 
    subset='description'
)

,name,description,type,unit,in_sample
0,source_id,Unique source identifier. Note that this *cannot* be matched against the DR1 or DR2 source_ids.,long,nan,True
1,r_med_geo,The median of the geometric distance posterior. The geometric distance estimate.,float,pc,True
2,r_lo_geo,The 16th percentile of the geometric distance posterior. The lower 1-sigma-like bound on the confidence interval.,float,pc,True
3,r_hi_geo,The 84th percentile of the geometric distance posterior. The upper 1-sigma-like bound on the confidence interval.,float,pc,True
4,r_med_photogeo,The median of the photogeometric distance posterior. The photogeometric distance estimate.,float,pc,True
5,r_lo_photogeo,The 16th percentile of the photogeometric distance posterior. The lower 1-sigma-like bound on the confidence interval.,float,pc,True
6,r_hi_photogeo,The 84th percentile of the photogeometric distance posterior. The upper 1-sigma-like bound on the confidence interval.,float,pc,True
7,flag,Additional information on the solution. Do not use for filtering (see table note in the reference URL).,char,nan,True


### Download Gaia DR3 and Bailer-Jones distance data for the sample

In [39]:
cols = ", ".join(['gaia.'+c for c in COLUMNS])
query = f"""
SELECT TOP {N_SAMPLE} {cols}, bj.*
FROM {TABLE} as gaia
JOIN {TABLE_BJ} as bj
ON gaia.source_id = bj.source_id
WHERE gaia.random_index < {N_SAMPLE}
"""

job = Gaia.launch_job_async(query)
results = job.get_results()
print(f"Downloaded {len(results)} rows")
results

INFO: Query finished. [astroquery.utils.tap.core]
Downloaded 97151 rows


source_id,ra,ra_error,dec,dec_error,parallax,parallax_error,parallax_over_error,pm,pmra,pmra_error,pmdec,pmdec_error,phot_g_mean_flux,phot_g_mean_flux_error,phot_g_mean_flux_over_error,phot_g_mean_mag,phot_bp_mean_flux,phot_bp_mean_flux_error,phot_bp_mean_flux_over_error,phot_bp_mean_mag,phot_rp_mean_flux,phot_rp_mean_flux_error,phot_rp_mean_flux_over_error,phot_rp_mean_mag,bp_rp,bp_g,g_rp,radial_velocity,radial_velocity_error,phot_variable_flag,l,b,ecl_lon,ecl_lat,in_qso_candidates,in_galaxy_candidates,non_single_star,teff_gspphot,teff_gspphot_lower,teff_gspphot_upper,logg_gspphot,logg_gspphot_lower,logg_gspphot_upper,mh_gspphot,mh_gspphot_lower,mh_gspphot_upper,distance_gspphot,distance_gspphot_lower,distance_gspphot_upper,source_id2,r_med_geo,r_lo_geo,r_hi_geo,r_med_photogeo,r_lo_photogeo,r_hi_photogeo,flag
,deg,mas,deg,mas,mas,mas,,mas / yr,mas / yr,mas / yr,mas / yr,mas / yr,electron / s,electron / s,,mag,electron / s,electron / s,,mag,electron / s,electron / s,,mag,mag,mag,mag,km / s,km / s,,deg,deg,deg,deg,,,,K,K,K,log(cm.s**-2),log(cm.s**-2),log(cm.s**-2),dex,dex,dex,pc,pc,pc,,pc,pc,pc,pc,pc,pc,
int64,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float64,float32,float64,float32,float32,float32,float64,float32,float32,float32,float64,float32,float32,float32,float32,float32,float32,float32,float32,object,float64,float64,float64,float64,bool,bool,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int64,float32,float32,float32,float32,float32,float32,object
29248903380172288,45.20367801982682,0.6059415,14.140665760221435,0.5559801,0.656781759691329,0.71180177,0.9227032,7.8249893,-6.637735769403847,0.8669393,4.143781434213999,0.8007746,240.87029703744915,1.3450699,179.07642,19.732908,85.58494032788751,10.595634,8.077377,20.50755,532.451414722924,13.299519,40.03539,17.932196,2.5753536,0.77464104,1.8007126,--,--,NOT_AVAILABLE,164.2097111374017,-38.150113925065575,46.83672812901629,-2.8414297836624627,False,False,0,--,--,--,--,--,--,--,--,--,--,--,--,29248903380172288,1367.0315,866.4941,2117.1895,818.22284,699.63116,972.65076,10033
52877064665219968,62.989185320237354,0.5176345,22.85389713641393,0.27764824,0.07599107563831119,0.5871929,0.12941417,2.108431,2.0536361372496157,0.64261806,0.4775569560708776,0.41455373,229.63781757264644,0.9831039,233.58449,19.784758,73.9585398176626,9.899422,7.470996,20.66607,255.65602901184138,11.98675,21.32822,18.728756,1.937315,0.8813133,1.0560017,--,--,NOT_AVAILABLE,171.92189536143854,-20.404447621448526,65.24798266232801,1.7057364954990877,False,False,0,--,--,--,--,--,--,--,--,--,--,--,--,52877064665219968,2324.696,1445.2452,3147.6804,2176.4417,1872.0151,2454.8804,10033
59112012852503936,49.29997591112532,0.16652386,18.312813562434645,0.14102176,0.0965757498599806,0.18283027,0.52822626,9.291811,8.259130951212054,0.2072811,-4.257288074327876,0.17655478,702.8755193864476,1.5501852,453.4139,18.570171,388.07480562403964,13.136911,29.540794,18.866253,484.2550392246904,13.77851,35.145676,18.03521,0.83104324,0.29608154,0.5349617,--,--,NOT_AVAILABLE,164.85889176318733,-32.44161436280661,51.75142941729137,0.11354724029262889,False,False,0,5413.49,5387.507,5470.1226,4.729,4.7065,4.7521,-2.9793,-3.603,-2.5077,2856.9841,2797.9998,2988.5073,59112012852503936,3061.168,2238.5342,4513.4263,6138.303,5035.58,7445.4272,10033
64862978357915648,56.12971135345475,0.35191908,22.89337583115117,0.23541677,0.36762551559993173,0.36883834,0.9967118,5.208627,-0.6171627149357772,0.40023443,-5.171934470905391,0.27888617,328.44711837565336,1.3611165,241.30713,19.396204,131.60214050401115,12.572789,10.467219,20.040384,368.4338096586582,10.843221,33.978264,18.331997,1.7083874,0.6441803,1.0642071,--,--,NOT_AVAILABLE,167.01087369254824,-24.8090656794721,59.06079685788747,3.018438597408884,False,False,0,--,--,--,--,--,--,--,--,--,--,--,--,64862978357915648,1842.8903,1183.6193,2846.8853,2258.5808,1982.7019,2589.6824,10033
65129987883239552,55.592

In [40]:
results = results.to_pandas()

In [44]:
dr3_sample  = 'dr3_sample_v2'

dp.save(
    results,
    name=dr3_sample,
    type='data',
    fmt='parquet',
    tags=['dr3', 'sample', 'bailer-jones'],
    notes=f'{len(results)} random rows from {TABLE} crossmached with {TABLE_BJ} to get Bailer-Jones distance estimates.\
    Columns: {COLUMNS} plus all columns from {TABLE_BJ}.',
)

{'type': 'data',
 'root': 'data',
 'path': 'dr3_sample_v2.parquet',
 'format': 'parquet',
 'updated_at': '2026-04-08T17:20:02+02:00',
 'updated_by': 'alex',
 'hash': 'sha256:5cfbfa9bc9db7c7d29830ee955629de9d8a4e58dce7f9b3be4f7d954ba8f8da6',
 'tags': ['bailer-jones', 'dr3', 'sample'],
 'inputs': [],
 'notes': "97151 random rows from gaiadr3.gaia_source crossmached with external.gaiaedr3_distance to get Bailer-Jones distance estimates.    Columns: ['source_id', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax', 'parallax_error', 'parallax_over_error', 'pm', 'pmra', 'pmra_error', 'pmdec', 'pmdec_error', 'phot_g_mean_flux', 'phot_g_mean_flux_error', 'phot_g_mean_flux_over_error', 'phot_g_mean_mag', 'phot_bp_mean_flux', 'phot_bp_mean_flux_error', 'phot_bp_mean_flux_over_error', 'phot_bp_mean_mag', 'phot_rp_mean_flux', 'phot_rp_mean_flux_error', 'phot_rp_mean_flux_over_error', 'phot_rp_mean_mag', 'bp_rp', 'bp_g', 'g_rp', 'radial_velocity', 'radial_velocity_error', 'phot_variable_flag', 'l', '

Notes on data volumes: the 10000 rows with 49 columns take ~3MB -> 2 billion of objects with the same number of columns should take 3 Mb/ (10\**4) objects in the sample \* (2\*10\**9) objects in DR3 / (10\**6) Mb ~ 0.6 TB

(But someone should recheck the math after me)